<a href="https://colab.research.google.com/github/pamepg23/AI-Revenue-Consultant-para-Airbnb/blob/main/rag_gemini_airbnb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 1. Instalar dependencias
# =========================
!pip install -qU langchain langchain-google-genai langchain-chroma chromadb gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2

In [4]:
# =========================
# 2. Configurar API Key
# =========================
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("PG_API Key")


PG_API Key··········


In [6]:
!pip install -qU langchain-text-splitters


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [8]:
# =========================
# 3. Imports
# =========================
import glob
import os
import shutil
import gradio as gr

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [45]:
import os
import shutil

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

os.makedirs(CHROMA_DIR, exist_ok=True)

print("Carpeta lista:", CHROMA_DIR)


Carpeta lista: /content/chroma_db_nueva


In [52]:
# =========================
# 4. Configuración general
# =========================
TXT_FOLDER = "/content/"
CHROMA_DIR = "/content/chroma_db_final"


MODEL_NAME = "gemini-2.5-flash"
EMBEDDING_MODEL = "models/gemini-embedding-001"


CHUNK_SIZE = 1000
CHUNK_OVERLAP = 180
TOP_K = 4


In [36]:
!rm -rf /content/docs_repo
!git clone https://github.com/pamepg23/AI-Revenue-Consultant-para-Airbnb.git /content/docs_repo


Cloning into '/content/docs_repo'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 19 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 14.82 KiB | 14.82 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [37]:
!pip install -q striprtf



In [38]:
from striprtf.striprtf import rtf_to_text


In [39]:
import glob
import os

rtf_files = sorted(glob.glob("/content/docs_repo/**/*.rtf", recursive=True))

print(f"Archivos .rtf encontrados: {len(rtf_files)}")

for file in rtf_files:
    print("-", file)


Archivos .rtf encontrados: 10
- /content/docs_repo/Anatomía de una Foto de Portada de Alto CTR.rtf
- /content/docs_repo/CONSEJOS REALES PARA PRECIOS EN AIRBNB.rtf
- /content/docs_repo/Checklist de Mantenimiento Preventivo.rtf
- /content/docs_repo/Cuadro de Verificación de Amenidades (Restocking) .rtf
- /content/docs_repo/Errores de Precio y Finanzas.rtf
- /content/docs_repo/Errores en el Anuncio.rtf
- /content/docs_repo/Estrategia de Precios para Temporada Alta.rtf
- /content/docs_repo/Guía Maestra: Consejos para un Anfitrión de Airbnb Exitoso (2026).rtf
- /content/docs_repo/La Estrategia del "Descuento de Lanzamiento".rtf
- /content/docs_repo/Manual de Estándares de Limpieza .rtf


In [40]:
!pip install -q striprtf


In [54]:
def limpiar_texto(texto):
    if texto is None:
        return ""

    # Elimina caracteres Unicode inválidos/surrogates
    texto = texto.encode("utf-8", "ignore").decode("utf-8", "ignore")

    # Limpieza básica
    texto = texto.replace("\x00", " ")
    texto = " ".join(texto.split())

    return texto


In [55]:
documents = []

for file_path in rtf_files[:10]:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        rtf_content = f.read()

    text = rtf_to_text(rtf_content)
    text = limpiar_texto(text)

    source_name = os.path.basename(file_path)

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": limpiar_texto(source_name),
                "path": file_path
            }
        )
    )

print(f"Documentos RTF limpios cargados: {len(documents)}")


Documentos RTF limpios cargados: 10


In [56]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks creados: {len(chunks)}")


Chunks creados: 18


In [57]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_rtf_documents_final"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": TOP_K}
)

print("ChromaDB creada correctamente en memoria.")


ChromaDB creada correctamente en memoria.


In [58]:
# =========================
# 8. Crear modelo Gemini y prompt RAG
# =========================
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.2
)

prompt = ChatPromptTemplate.from_template("""
Eres un asistente RAG experto. Responde únicamente con base en el contexto proporcionado.

Reglas obligatorias:
1. Si la respuesta está en el contexto, responde de forma clara y precisa.
2. Si no encuentras la respuesta en el contexto, di: "No encontré esa información en los documentos cargados."
3. Siempre incluye una sección final llamada "Fuentes".
4. En "Fuentes", menciona obligatoriamente el nombre del documento o documentos usados.
5. No inventes fuentes.

Contexto:
{context}

Pregunta:
{question}

Respuesta:
""")

chain = prompt | llm | StrOutputParser()


In [59]:
# =========================
# 9. Funciones del chat
# =========================
def format_docs(retrieved_docs):
    formatted = []

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("source", "Fuente desconocida")
        content = doc.page_content

        formatted.append(
            f"[Documento {i}: {source}]\n{content}"
        )

    return "\n\n".join(formatted)


def get_sources(retrieved_docs):
    sources = []

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Fuente desconocida")
        if source not in sources:
            sources.append(source)

    return sources


def rag_chat(message, history):
    retrieved_docs = retriever.invoke(message)

    if not retrieved_docs:
        return "No encontré información relevante en los documentos cargados.\n\nFuentes: Ninguna"

    context = format_docs(retrieved_docs)
    sources = get_sources(retrieved_docs)

    response = chain.invoke({
        "context": context,
        "question": message
    })

    # Garantía extra: si el modelo olvidó citar fuentes, las agregamos manualmente.
    if "Fuentes" not in response:
        response += "\n\nFuentes:\n" + "\n".join(f"- {source}" for source in sources)

    return response


In [71]:
# =========================
# 10. Interfaz Gradio
# =========================

# Creamos el tema claro personalizado
airbnb_light_theme = gr.themes.Soft(
    primary_hue="red",       # Para los acentos y botones
    neutral_hue="slate",     # Para el texto y bordes (un gris elegante)
    spacing_size="md",       # Espaciado equilibrado
    radius_size="lg",        # Bordes muy redondeados como la app de Airbnb
).set(
    # Fondo de la página en blanco puro
    body_background_fill="#FFFFFF",
    body_background_fill_dark="#1A1A1A", # Para quienes tengan modo oscuro

    # Botones principales en el rojo de Airbnb
    button_primary_background_fill="#FF385C",
    button_primary_background_fill_hover="#E31C5F",
    button_primary_text_color="#FFFFFF",

    # Input de texto (donde el usuario escribe)
    input_background_fill="#F7F7F7", # Gris muy claro casi blanco
    input_border_color="#DDDDDD",
)

# Definimos el ChatInterface usando la función rag_chat
chat_interface = gr.ChatInterface(
    fn=rag_chat,
    title="🏨 Airbnb Strategic Consultant",
    description="Analista de Revenue Management con soporte RAG.",
    chatbot=gr.Chatbot(height=500),
    textbox=gr.Textbox(
        placeholder="Pregúntame sobre precios, SEO de tu anuncio o cómo mejorar tus reseñas.",
        container=False,
        scale=7
    ),
    examples=[
        "Soy un anfitrión nuevo y no tengo reseñas, ¿qué estrategia de precios debo seguir las primeras semanas?",
        "¿Qué palabras clave o amenidades debería resaltar en mi título para atraer a nómadas digitales?",
        "¿Cómo debo ajustar mi tarifa si se acerca un evento importante en mi ciudad y mi ocupación aún es baja?"
    ]
)

# Lanzamos la interfaz aplicando el tema personalizado con gr.Blocks
with gr.Blocks(theme=airbnb_light_theme) as demo:
    chat_interface.render()

demo.launch(debug=True, share=True)

/tmp/ipykernel_11611/517392842.py:45: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=airbnb_light_theme) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e3323a199cbdb903e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 